In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
books = pd.read_csv(
    "BX-Books.csv",
    sep=";",
    encoding="latin-1",
    on_bad_lines="skip"
)

In [ ]:
books.head()

In [ ]:
books.iloc[0]['Image-URL-L']

In [ ]:
books.shape


In [ ]:
books.rename(
    columns={
        "Book-Title": "title",
        "Book-Author": "author",
        "Year-Of-Publication": "year",
        "Publisher": "publisher",
        "Image-URL-L": "image_url"
    },
    inplace=True
)

In [ ]:
user = pd.read_csv(
    "BX-Users.csv",
    sep=";",
    encoding="latin-1",
    on_bad_lines="skip"
)

In [ ]:
user.head()

In [ ]:
user.shape

In [ ]:
user.rename(columns={"User-ID":"user_id","Location":"location","Age":"age"},inplace=True)

In [ ]:
ratings=pd.read_csv("BX-Book-Ratings.csv",sep=";",encoding="latin-1",on_bad_lines="skip")

In [ ]:
ratings.head()

In [ ]:
ratings.rename(columns={"User-ID":"user_id","Book-Rating":"rating"},inplace=True)

In [ ]:
ratings.head()

In [ ]:
print(books.shape,user.shape,ratings.shape,sep="\n")

In [ ]:
ratings["user_id"].value_counts()

In [ ]:
x=ratings['user_id'].value_counts()>200

In [ ]:
x

In [ ]:
x[x].shape

In [ ]:
y=x[x].index

In [ ]:
ratings=ratings[ratings['user_id'].isin(y)]
ratings.head()

In [ ]:
ratings_with_books = ratings.merge(books, on='ISBN')

In [ ]:
ratings_with_books.head()

In [ ]:
number_rating = ratings_with_books.groupby('title')['rating'].count().reset_index()

In [ ]:
number_rating.head()

In [ ]:
number_rating.rename(columns={"rating":"number_of_rating"},inplace=True)

In [ ]:
number_rating.head()

In [ ]:
final_rating=ratings_with_books.merge(number_rating,on='title')
final_rating.head()

In [ ]:
final_rating.head()

In [ ]:
final_rating.drop_duplicates(['user_id','title'],inplace=True)
final_rating.head()

In [ ]:
final_rating=final_rating[final_rating['number_of_rating'] >=50]
final_rating.head()

In [ ]:
final_rating.isnull().sum()

In [ ]:
book_pivot = final_rating.pivot_table(
    columns='user_id',
    index='title',
    values='rating'
)

In [ ]:
book_pivot.fillna(0, inplace=True)


In [ ]:
similarity_scores = cosine_similarity(book_pivot)


In [ ]:
import pickle
import os
os.makedirs('../artifacts', exist_ok=True)

# Save full books (to match previous artifacts behavior)
books_for_pkl = books.rename(columns={'ISBN': 'isbn'})[[
    'isbn', 'title', 'author', 'year', 'publisher', 'image_url'
]].copy()

pickle.dump(books_for_pkl, open('../artifacts/books.pkl', 'wb'))
pickle.dump(book_pivot, open('../artifacts/pt.pkl', 'wb'))
pickle.dump(similarity_scores, open('../artifacts/similarity_scores.pkl', 'wb'))

print('Artifacts saved (full books.pkl to stay compatible)')
print('books.pkl shape:', books_for_pkl.shape)


In [ ]:
def recommend(book_name, top_n=5):
    if book_name not in book_pivot.index:
        matches = [title for title in book_pivot.index if book_name.lower() in title.lower()][:10]
        print(f'Book not found in pivot table: {book_name}')
        if matches:
            print('Did you mean one of these?')
            for title in matches:
                print('-', title)
        return []

    index = np.where(book_pivot.index == book_name)[0][0]
    similar_items = sorted(
        list(enumerate(similarity_scores[index])),
        key=lambda x: x[1],
        reverse=True
    )[1:top_n + 1]

    recommendations = [book_pivot.index[i[0]] for i in similar_items]
    for title in recommendations:
        print(title)
    return recommendations

recommend('4 Blondes')
